In [ ]:
import os
import numpy as np
import torch
from argparse import ArgumentParser
import tqdm 

from config import *
from helpers import * 
import articulate as art
from constants import MODULES
from utils.model_utils import load_model
from data import PoseDataset
from models import MobilePoserNet


class PoseEvaluator:
    def __init__(self):
        self._eval_fn = art.FullMotionEvaluator(paths.smpl_file, joint_mask=torch.tensor([2, 5, 16, 20]), fps=datasets.fps)

    def eval(self, pose_p, pose_t, joint_p=None, tran_p=None, tran_t=None):
        pose_p = pose_p.clone().view(-1, 24, 3, 3)
        pose_t = pose_t.clone().view(-1, 24, 3, 3)
        print("the pose: ", pose_p.shape,pose_t.shape)
        tran_p = tran_p.clone().view(-1, 3)
        tran_t = tran_t.clone().view(-1, 3)
        pose_p[:, joint_set.ignored] = torch.eye(3, device=pose_p.device)
        pose_t[:, joint_set.ignored] = torch.eye(3, device=pose_t.device)

        errs = self._eval_fn(pose_p, pose_t, tran_p=tran_p, tran_t=tran_t)
        return torch.stack([errs[9], errs[3], errs[9], errs[0]*100, errs[7]*100, errs[1]*100, errs[4] / 100, errs[6]])

    @staticmethod
    def print(errors):
        for i, name in enumerate(['SIP Error (deg)', 'Angular Error (deg)', 'Masked Angular Error (deg)',
                                  'Positional Error (cm)', 'Masked Positional Error (cm)', 'Mesh Error (cm)', 
                                  'Jitter Error (100m/s^3)', 'Distance Error (cm)']):
            print('%s: %.2f (+/- %.2f)' % (name, errors[i, 0], errors[i, 1]))


@torch.no_grad()
def evaluate_pose(model, dataset, num_past_frame=20, num_future_frame=5, evaluate_tran=False):
    # specify device
    print("the evaluate train is: ",evaluate_tran)
    # print(getenv("ONLINE"))
    device = model_device

    # load data
    xs, ys = zip(*[(imu.to(device), (pose.to(device), tran)) for imu, pose, joint, tran in dataset])

    # setup Pose Evaluator
    evaluator = PoseEvaluator()

    # track errors
    offline_errs, online_errs = [], []
    tran_errors = {window_size: [] for window_size in list(range(1, 8))}
    
    model.eval()
    with torch.no_grad():
        for x, y in tqdm.tqdm(list(zip(xs, ys))):
            model.reset()
            pose_p_offline, joint_p_offline, tran_p_offline, _ = model.forward_offline(x.unsqueeze(0), [x.shape[0]])
            pose_t, tran_t = y
            pose_t = art.math.r6d_to_rotation_matrix(pose_t)

            offline_errs.append(evaluator.eval(pose_p_offline, pose_t, tran_p=tran_p_offline, tran_t=tran_t))


    # print joint errors
    print('============== offline ================')
    evaluator.print(torch.stack(offline_errs).mean(dim=0))
    if getenv("ONLINE"):
        print('============== online ================')
        evaluator.print(torch.stack(online_errs).mean(dim=0))
    
    # print translation errors
    if evaluate_tran:
        print([0] + [torch.tensor(_).mean() for _ in tran_errors.values()])




In [29]:
model_path = r"C:\deepika\2_mobileposer\MobilePoser\mobileposer\deepika\model_finetuned.pth"
dataset_name = "dip"

model = load_model(model_path)

if dataset_name not in datasets.test_datasets:
    raise ValueError(f"Test dataset: {dataset_name} not found.")

dataset = PoseDataset(fold="test", evaluate=dataset_name)

print(f"Starting evaluation: {dataset_name.capitalize()}")

results = evaluate_pose(
    model,
    dataset
)


c:\deepika\2_mobileposer\MobilePoser\mobileposer\utils\model_utils.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_loca

Starting evaluation: Dip
the evaluate train is:  False


  0%|          | 0/228 [00:00<?, ?it/s]

the pose:  torch.Size([2373, 24, 3, 3]) torch.Size([2373, 24, 3, 3])


  0%|          | 1/228 [00:02<09:00,  2.38s/it]

...

the pose:  torch.Size([2373, 24, 3, 3]) torch.Size([2373, 24, 3, 3])


  1%|▏         | 3/228 [00:07<09:22,  2.50s/it]

the pose:  torch.Size([2373, 24, 3, 3]) torch.Size([2373, 24, 3, 3])


  1%|▏         | 3/228 [00:09<11:50,  3.16s/it]


KeyboardInterrupt: 

In [4]:
model_path = r"C:\deepika\2_mobileposer\MobilePoser\mobileposer\deepika\model_finetuned.pth"
dataset_name = "dip"

model = load_model(model_path)

if dataset_name not in datasets.test_datasets:
    raise ValueError(f"Test dataset: {dataset_name} not found.")

dataset = PoseDataset(fold="test", evaluate=dataset_name)

100%|██████████| 1/1 [00:00<00:00,  6.00it/s]


In [7]:
dataset

In [ ]:
xs, ys = zip(*[(imu.to(device), (pose.to(device), tran)) for imu, pose, joint, tran in dataset])

In [ ]:
i = 0
for imu, pose, joint, tran in dataset:
    if i < 1:
        print(pose.shape)
        i+=1 
        print(i)
    else :
        break

torch.Size([2373, 144])
1


In [ ]:
device = model_device

# load data
xs, ys = zip(*[(imu.to(device), (pose.to(device), tran)) for imu, pose, joint, tran in dataset])

# setup Pose Evaluator
# evaluator = PoseEvaluator()

# track errors
offline_errs, online_errs = [], []
tran_errors = {window_size: [] for window_size in list(range(1, 8))}

model.eval()
with torch.no_grad():
    for x, y in tqdm.tqdm(list(zip(xs, ys))):
        model.reset()
        pose_p_offline, joint_p_offline, tran_p_offline, _ = model.forward_offline(x.unsqueeze(0), [x.shape[0]])
        pose_t, tran_t = y
        
        pose_t_rotational_matrix = art.math.r6d_to_rotation_matrix(pose_t)


100%|██████████| 228/228 [01:29<00:00,  2.53it/s]


In [46]:
print(pose_t.shape)

torch.Size([2708, 144])


In [47]:
print(tran_t.shape)

torch.Size([2708, 3])


In [25]:
print(pose_p_offline.shape)

torch.Size([2708, 24, 3, 3])


In [26]:
print(pose_t_rotational_matrix.shape)

torch.Size([64992, 3, 3])


In [36]:
# pose_p_offline.shape(2708,-1)
# pose_p_offline.shape(2708,-1)
pose_p_offline = pose_p_offline.clone().view(2708, -1)
print(pose_p_offline.shape)

# C = np.hstack((pose_t,tran_))  # shape will be (2708, 147)

# print(C.shape)

torch.Size([2708, 216])


In [37]:
tran_p_offline = tran_p_offline.clone().view(2708, -1)
print(tran_p_offline.shape)
 

torch.Size([2708, 3])


In [40]:
joint_p_offline = joint_p_offline.clone().view(2708,-1)
print(joint_p_offline.shape)

torch.Size([2708, 72])


In [42]:
type(pose_p_offline)

torch.Tensor

In [ ]:
class Block(nn.Module):
    def __init__(self, size: int):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.ff = nn.Linear(size, size)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor):
        return x + self.act(self.ff(self.ln(x)))

class SinusoidalEmbedding(nn.Module):
    def __init__(self, size: int, scale: float = 1.0):
        super().__init__()
        self.size = size
        self.scale = scale
        half_size = self.size // 2
        emb = torch.log(torch.Tensor([10000.0])) / (half_size - 1)
        emb = torch.exp(-emb * torch.arange(half_size))
        self.emb = nn.Parameter(emb, requires_grad=False)

    def forward(self, x: torch.Tensor):
        x = x * self.scale
        emb = x * self.emb.unsqueeze(0)
        emb = torch.cat((torch.sin(emb), torch.cos(emb)), dim=-1)
        return emb

class MLP(nn.Module):
    def __init__(self, hidden_size: int = 128, hidden_layers: int = 3, emb_size: int = 128,
                 twoD_data: bool = True):
        super().__init__()
        self.twoD_data = twoD_data
        self.time_emb = SinusoidalEmbedding(emb_size)
        if twoD_data:
            self.input_emb1 = SinusoidalEmbedding(emb_size, scale=25.0)
            self.input_emb2 = SinusoidalEmbedding(emb_size, scale=25.0)
            self.concat_size = 2 * emb_size + emb_size # 2d concat time
            self.data_size = 2
        else:
            self.concat_size = 28 * 28 + emb_size # mnist is 28*28
            self.data_size = 28 * 28

        layers = [nn.Linear(self.concat_size, hidden_size), nn.GELU()]
        for _ in range(hidden_layers):
            layers.append(Block(hidden_size))
        layers.append(nn.LayerNorm(hidden_size))
        layers.append(nn.Linear(hidden_size, self.data_size))
        self.joint_mlp = nn.Sequential(*layers)

    def forward(self, x, t):
        t_emb = self.time_emb(t.reshape(-1,1))
        if self.twoD_data:
            x1_emb = self.input_emb1(x[:, 0].reshape(-1,1))
            x2_emb = self.input_emb2(x[:, 1].reshape(-1,1))
            x = torch.cat((x1_emb, x2_emb), dim=-1)
        x = torch.cat((x, t_emb), dim=-1)
        x = self.joint_mlp(x)
        return x

class Diffusion():
    def __init__(self,
                 num_timesteps=1000,
                 beta_start=0.0001,
                 beta_end=0.02):

        self.num_timesteps = num_timesteps
        self.betas = torch.linspace(
                beta_start, beta_end, num_timesteps, dtype=torch.float32)

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(
            self.alphas_cumprod[:-1], (1, 0), value=1.)

        self.sqrt_alphas_cumprod = self.alphas_cumprod ** 0.5
        self.sqrt_one_minus_alphas_cumprod = (1 - self.alphas_cumprod) ** 0.5

        self.sqrt_inv_alphas = torch.sqrt(1. / self.alphas)
        self.noise_coef = self.betas / self.sqrt_one_minus_alphas_cumprod
        self.variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    def add_noise(self, x_start, x_noise, timesteps):
        # compute x_t for DDPM algorithm 1
        s1 = self.sqrt_alphas_cumprod[timesteps].reshape(-1, 1)
        s2 = self.sqrt_one_minus_alphas_cumprod[timesteps].reshape(-1, 1)
        return s1 * x_start + s2 * x_noise
    
    def sample_step(self, model_output, timestep, sample):
        # compute x_{t-1} for DDPM algorithm 2
        s1 = self.sqrt_inv_alphas[timestep].reshape(-1, 1)
        s2 = self.noise_coef[timestep].reshape(-1, 1)
        s3 = self.variance[timestep].reshape(-1, 1) ** 0.5
        noise = torch.randn_like(model_output)
        return s1 * (sample - s2 * model_output) + s3 * noise

    def __len__(self):
        return self.num_timesteps